In [ ]:
import os
from pathlib import Path
import boto3
from data_profiling import ProfileReport

# HTML profile reports are saved here
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Credentials come from the "factored" profile in ~/.aws/credentials
# (set it up once with: aws configure --profile factored). Never paste keys here.
session = boto3.Session(
    #profile_name=os.getenv("AWS_PROFILE", "factored"),
    region_name="us-east-2",
)
print(session.region_name)

# Create client
s3 = session.client("s3")
bucket = "factored-datathon-2026-s3-157725502942-us-east-2-an"

# List all available data files (paginated: list_objects_v2 returns max 1000 keys per call)
paginator = s3.get_paginator("list_objects_v2")
keys = [
    obj["Key"]
    for page in paginator.paginate(Bucket=bucket, Prefix="data/")
    for obj in page.get("Contents", [])
]
for key in keys:
    print(key)

In [ ]:
import pandas as pd

table_keys = {
    "branches": "data/branches.csv",
    "customers": "data/customers.csv",
    "products": "data/products.csv",
    "service_agents": "data/service_agents.csv",
}

missing_keys = [key for key in table_keys.values() if key not in keys]
if missing_keys:
    raise FileNotFoundError(f"CSV files not found in S3: {missing_keys}")

tables = {
    name: pd.read_csv(s3.get_object(Bucket=bucket, Key=key)["Body"])
    for name, key in table_keys.items()
}

branches = tables["branches"]
customers = tables["customers"]
products = tables["products"]
service_agents = tables["service_agents"]

for name, dataframe in tables.items():
    print(f"{name}: {dataframe.shape[0]:,} rows, {dataframe.shape[1]} columns")

In [ ]:
profile = ProfileReport(customers, title="Customers EDA Report")
profile.to_file(REPORTS_DIR / "customers_eda.html")
profile.to_notebook_iframe()

In [ ]:
profile = ProfileReport(products, title="Products EDA Report")
profile.to_file(REPORTS_DIR / "products_eda.html")
profile.to_notebook_iframe()

In [ ]:
profile = ProfileReport(branches, title="Branches EDA Report")
profile.to_file(REPORTS_DIR / "branches_eda.html")
profile.to_notebook_iframe()

In [ ]:
profile = ProfileReport(service_agents, title="Service agents EDA Report")
profile.to_file(REPORTS_DIR / "aservice_agents_eda.html")
profile.to_notebook_iframe()